<img src="https://raw.githubusercontent.com/AllenSWDB/allenswdb.github.io/main/databook/resources/swdb_logo_new.jpg">  

# Part 3 — Simulating the analysis **with a real effect**


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; color: #000000;">


In Part 1 we developed the proposed workflow and tested it on data with **no** real effect.
In Part 2 we refined the analysis to address its flaws (train/test split and multiple-comparison correction).
Now that we have an analysis workflow that does not generate spurious false positives, we can ask whether 
the proposed analysis could detect a real effect if one were present.

For this we repeat the simulation, but this time simulate that behavioral performance really *does* modulate the
activity of certain **performance-correlated cells** (we simulate that **10% of the cells** are
genuinely modulated by behavioral performance). We will simulate that by adding, on top of the noise, a
signal that is a scaled, mean-zero function of each trial group's performance $P$:

$$P_\text{scaled} = \frac{k\, \sigma_A}{\operatorname{std}(P)}\,(P - \overline{P})$$

The constant `k` sets how big the behavioral effect is relative to noise:
`k=1` → the signal is comparable to the noise; `k=0.1` → performance only weakly affects it.
Here `k=0.2` gives a modest effect size. All other cells remain pure noise. 
The refined analysis workflow is applied exactly as in Part 2.

This notebook is self-contained: the shared building blocks from Parts 1 and 2 are imported
from `utils.py`, so it can be run on its own.

</div>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

import sys
sys.path.append("/code")
from utils import (
    simulate_null_session,
    correlate_and_select,
    annotate_corr,
    train_test_PRCA,
    fdr_bh_cells,
)

# Set a seed so the notebook is reproducible (remove for fresh random draws)
rng = np.random.default_rng(0)

Ngroups, Ncells = 20, 200
mean_perf, sd_perf, sd_dff = 75, 8, 1

# Recreate the Part 1 null session (behavior, activity, cell positions).
Perf, DA, CellXY = simulate_null_session(rng, Ngroups, Ncells, mean_perf, sd_perf, sd_dff)

# Part 1 circular workflow on the null data
r, p, SelectedCells = correlate_and_select(DA, Perf)
PRCA = DA[:, SelectedCells].mean(axis=1)

# Part 2 train/test split on the null data
PRCA_test_null, Perf_test_null, _ = train_test_PRCA(DA, Perf, rng=rng)


In [ ]:
Nresponsive = round(0.1 * Ncells)
# Indices of the cells that are genuinely modulated by performance (only in the real-effect case).
CorrelatedCells = np.arange(0, Nresponsive)
UncorrelatedCells = np.arange(Nresponsive, Ncells)


In [ ]:
k = 0.2  # modest correlation between cell signal and performance
Pscaled = (k * sd_dff / np.std(Perf, ddof=1)) * (Perf - np.mean(Perf))

DA_real = sd_dff * rng.standard_normal((Ngroups, Ncells))
DA_real[:, CorrelatedCells] += Pscaled[:, np.newaxis]

r2 = np.empty(Ncells)
p2 = np.empty(Ncells)
for j in range(Ncells):
    r2[j], p2[j] = stats.pearsonr(DA_real[:, j], Perf)
SelectedCells_real = np.where((r2 > 0.1) & (p2 < 0.05))[0]
PRCA_real = DA_real[:, SelectedCells_real].mean(axis=1)
print(f"Number of 'selected' cells: {SelectedCells_real.size}")

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Task 3.1:** How can you check that 10% of neurons now have a real effect?

</div>


In [ ]:
#:


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; color: #000000;">

## Plot the real-effect result for the original workflow

First run the originally proposed (circular) analysis on the real-effect data and plot PRCA vs.
performance. This time there genuinely is an effect so a good analysis should find it.

</div>


<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Task 3.2:** Recall that in notebook 1 we made a helper function `annotate_corr` that ran the originally proposed circular workflow. Use that helper function to see what result that method produces now that there is a real result.
</div>


In [ ]:
#:

<div style="border-left: 3px solid #f0b429; padding: 1px; padding-left: 10px; background: #FFF9C4;  max-width: 90%; overflow-x: auto; color: #000000;">

<details>
<summary><b>Expected outcome:</b></summary>

There is a real effect in this case — but note that this real-effect plot looks much
like the null one, which is exactly why we can't trust the circular sampling method.
</details>
<br>
</div>


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; color: #000000;">

## Did the analysis identify the *right* cells? (spatial view)

Because each cell has a random 2D position, we can plot the imaging field and highlight the
subset of cells that the circular workflow *selected* as correlated with performance. Here we
can compare this to the **real-effect** data: the truly performance-correlated cells (the first
`Nresponsive` cells) are outlined for reference, and the selected set should overlap them more
often than chance — though, since positions are random, the selection still looks spatially
unstructured.

</div>


In [ ]:
def plot_cells_2d(ax, selected, title, true_subset=None):
    """Plot all cells at their 2D positions, highlighting the `selected` subset."""
    ax.scatter(CellXY[:, 0], CellXY[:, 1], s=20, color="lightgray",
               label="all cells", zorder=1)
    if true_subset is not None:
        ax.scatter(CellXY[true_subset, 0], CellXY[true_subset, 1], s=90,
                   facecolors="none", edgecolors="tab:orange", linewidths=1.5,
                   label="truly correlated cells", zorder=2)
    ax.scatter(CellXY[selected, 0], CellXY[selected, 1], s=30, color="tab:red",
               label="selected cells", zorder=3)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_aspect("equal")
    ax.set_xlabel("x position")
    ax.set_ylabel("y position")
    ax.set_title(title)
    ax.legend(loc="upper right", fontsize=8)


fig, ax = plt.subplots(figsize=(7, 6))
plot_cells_2d(ax, SelectedCells_real,
              f"With Real Effect — {SelectedCells_real.size} selected cells",
              true_subset=CorrelatedCells)
plt.tight_layout()
plt.show()


<div style="border-left: 3px solid #f0b429; padding: 1px; padding-left: 10px; background: #FFF9C4;  max-width: 90%; overflow-x: auto; color: #000000;">

<details>
<summary><b>Expected outcome:</b></summary>
This plot shows that the circular analysis method is poor in two ways: a high proportion the selected cells don't have real effects (false positives), and many of the cells with real effects are not found (false negatives).
</details>
<br>
</div>



<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; color: #000000;">

## Do the refined (valid) methods detect the real effect?

In Part 2 we came up with refined methods with properly controlled false positive rates — the 50/50 train/test split and FDR-BH. Both refinements have the down-side that they reduced power for detecting true effects.  Now we can apply those to the **real-effect** data to see if real effects can still be found. Unlike the null case, here we *want* a detection. 

</div>


<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Task 3.3** Recall that in notebook 2 we wrote a helper function `train_test_PRCA` that ran a cross-validated version of the analysis, and another function `fdr_bh_cells` that ran a multiple-comparison-corrected version of the analysis, both of which solved the false positive problem. Use those to analyize the ground-truth real effect to see if the effect is detected.
</div>


In [ ]:
# 50/50 train/test split on the real-effect data
#: run train_test_PRCA(DA_real, Perf) and pearsonr
# split_real = 
# rr, pp = 

# print the result details
print("Train/test split on the real-effect data:")
print(f"cells selected on training half : {split_real[2].size}")
print(f"test-half correlation           : r = {rr:+.3f}, p = {pp:.3f}")


# FDR-BH on the real-effect data - how many cells are selected?
#: fdr_real = fdr_bh_cells(...



<div style="border-left: 3px solid #f0b429; padding: 1px; padding-left: 10px; background: #FFF9C4;  max-width: 90%; overflow-x: auto; color: #000000;">

<details>
<summary><b>Expected outcome:</b></summary>
The improved methods don't have the statistical power to detect a real effect of the size we simulated.
</details>
<br>
</div>



<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; color: #000000;">

### Stronger effects
The train/test split prevents us from detecting the false positive in the no-real-effect dataset, but with the weak effect we may also fail to see a true positive result. How strong would a real effect have to be, for us to detect it?

</div>


<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">
<p><b>Task 3.4:</b> Simulate a stronger real effect by raising <code>k</code> (try <code>k_strong = 0.6</code>), rebuild <code>DA_real_strong</code> the same way as <code>DA_real</code> above, then rerun the per-cell correlation and selection to get <code>SelectedCells_real_strong</code> and <code>PRCA_real_strong</code>. First we will run the flawed (circular) analysis:
</div>


In [ ]:
#: k_strong =



In [ ]:
# Train/test split on the stronger-effect data
#: split_real_strong =

# print some diagnostics
print("Weak real-effect data (train/test split):")
r_w, p_w = stats.pearsonr(split_real[0], split_real[1])
print(f"  cells selected on training half : {split_real[2].size}")
print(f"  test-half correlation           : r = {r_w:+.3f}, p = {p_w:.3f}")

print("\nStrong real-effect data (train/test split):")
r_s, p_s = stats.pearsonr(split_real_strong[0], split_real_strong[1])
print(f"  cells selected on training half : {split_real_strong[2].size}")
print(f"  test-half correlation           : r = {r_s:+.3f}, p = {p_s:.3f}")


In [ ]:
#: reuse plot_cells_2d to plot the selected cells vs real cells



In [ ]:
# Plot the train/test split results — no effect, weak effect, strong effect
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

panels = [
    (PRCA_test_null, Perf_test_null, "No Real Effect"),
    (split_real[0], split_real[1], "Weak Real Effect"),
    (split_real_strong[0], split_real_strong[1], "Strong Real Effect"),
]

for ax, (PRCA_panel, perf, label) in zip(axes, panels):
    annotate_corr(ax, PRCA_panel, perf, label)

plt.tight_layout()
plt.show()


<div style="border-left: 3px solid #f0b429; padding: 1px; padding-left: 10px; background: #FFF9C4;  max-width: 90%; overflow-x: auto; color: #000000;">

#### Conclusions

The refined methods don't generate excess false positive results, but they reduce statistical power. They can only detect rather strong effects, but if they do detect an effect, their result can be trusted. 

- In extension-1 you can explore more systematically the power to detect real effects of different sizes.You can also explore even better cross-validation methods (such as LOO-CV) that are more data efficient and therefore should have better statistical power.
- In extension-2 you can explore applying the refined anlaysis to real data